In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.regression.mixed_linear_model import MixedLM
from memoria_icm.datos import *
from memoria_icm.funciones import *
from memoria_icm.graficos import *

In [2]:
df, df_resumen = cargar_datos()

Datos cargados. Trackeo: (44324, 28), Resumen: (192, 42)


Este cuadernillo analiza la existencia de diferencias significativas entre los grupos de la misma especie. RECUPERACIÓN.

In [3]:
columnas = list(df_resumen.select_dtypes(include='float64').columns)
columnas.remove('duracion_video')
columnas

['r_promedio',
 'r_mediana',
 'r_centrado_promedio',
 'r_centrado_mediana',
 'v_promedio',
 'v_mediana',
 'a_promedio',
 'a_mediana',
 'dr_promedio',
 'dr_mediana',
 'dr_centrado_promedio',
 'dr_centrado_mediana',
 'theta_r_promedio',
 'theta_r_mediana',
 'theta_r_centrado_promedio',
 'theta_r_centrado_mediana',
 'v_angular_promedio',
 'v_angular_mediana',
 'v_angular_centrado_promedio',
 'v_angular_centrado_mediana',
 'a_angular_promedio',
 'a_angular_mediana',
 'a_angular_centrado_promedio',
 'a_angular_centrado_mediana',
 'curvatura_promedio',
 'curvatura_mediana',
 'theta_v_promedio',
 'theta_v_mediana']

In [4]:
cc = df_resumen[(df_resumen['especie'] == 'CC') & (df_resumen['tratamiento'] == 'R')] 
la = df_resumen[(df_resumen['especie'] == 'LA') & (df_resumen['tratamiento'] == 'R')]

In [5]:
print(cc['salinidad'].unique())
print(la['salinidad'].unique())

['35-N' '35-R' '40' '45' '50' '55' '60' '20' '25' '30' '35']
['35-N' '35-R' '40' '45' '50' '15' '20' '25' '30' '35']


In [5]:
grupos = {'CC': cc, 'LA': la}
resultados = {}

factor = 'nivel_salinidad'

for nombre_grupo, df in grupos.items():
    clave = f"{nombre_grupo}"
    resultados[clave] = pd.concat([testear_supuestos(df_resumen, var, factor) for var in columnas],
                                  ignore_index=True)
    
df_resultados = pd.concat([pd.DataFrame(lista).assign(grupo=clave) for clave, lista in resultados.items()],
                          ignore_index=True)

df_resultados

,variable,factor,grupo,test,p_value,cumple_supuesto,n,test_sugerido
0,r_promedio,nivel_salinidad,CC,Shapiro-Wilk,0.293732,True,70,Kruskal-Wallis
1,r_promedio,nivel_salinidad,CC,Shapiro-Wilk,0.010246,False,67,Kruskal-Wallis
2,r_promedio,nivel_salinidad,CC,Shapiro-Wilk,0.000005,False,55,Kruskal-Wallis
3,r_promedio,nivel_salinidad,CC,Levene,0.000039,False,192,Kruskal-Wallis
4,r_mediana,nivel_salinidad,CC,Shapiro-Wilk,0.298718,True,70,Kruskal-Wallis
...,...,...,...,...,...,...,...,...
219,theta_v_promedio,nivel_salinidad,LA,Levene,0.470140,True,192,Kruskal-Wallis
220,theta_v_mediana,nivel_salinidad,LA,Shapiro-Wilk,0.429376,True,70,ANOVA
221,theta_v_mediana,nivel_salinidad,LA,Shapiro-Wilk,0.440234,True,67,ANOVA
222,theta_v_mediana,nivel_salinidad,LA,Shapiro-Wilk,0.555174,True,55,ANOVA


## LOCOS

### Análisis de la varianza

In [7]:
resultados_cc = df_resultados[(df_resultados['grupo'] == 'CC')]

variables_resumen_por_test = (resultados_cc[['variable', 'test_sugerido']].drop_duplicates()
    .groupby('test_sugerido')['variable']
    .unique()
    .to_dict())

print("LOCOS - QUÉ APLICAR SEGÚN SUPUESTOS")
for tipo, lista in variables_resumen_por_test.items():
    print(f"Puedo aplicar {tipo} en: {list(lista)}")
    
#filtro
listas_por_test = {
    tipo: grupo['variable'].drop_duplicates().tolist()
    for tipo, grupo in resultados_cc.dropna(subset=['test_sugerido']).groupby('test_sugerido')}

#ver ranking
df_efectos_cc = analizar_grupo(cc, listas_por_test, 'nivel_salinidad')
pvalues_significativos_cc = df_efectos_cc.loc[(df_efectos_cc['p_value'] < 0.05)]
variables_candidatas_cc = pvalues_significativos_cc['variable'].tolist()
variables_candidatas_cc = set(variables_candidatas_cc) #podemos mostrar el dataframe completo también
print(pvalues_significativos_cc)

LOCOS - QUÉ APLICAR SEGÚN SUPUESTOS
Puedo aplicar ANOVA en: ['theta_v_mediana']
Puedo aplicar Kruskal-Wallis en: ['r_promedio', 'r_mediana', 'r_centrado_promedio', 'r_centrado_mediana', 'v_promedio', 'v_mediana', 'a_promedio', 'a_mediana', 'dr_promedio', 'dr_mediana', 'dr_centrado_promedio', 'dr_centrado_mediana', 'theta_r_promedio', 'theta_r_mediana', 'theta_r_centrado_promedio', 'theta_r_centrado_mediana', 'v_angular_promedio', 'v_angular_mediana', 'v_angular_centrado_promedio', 'v_angular_centrado_mediana', 'a_angular_promedio', 'a_angular_mediana', 'a_angular_centrado_promedio', 'a_angular_centrado_mediana', 'curvatura_promedio', 'curvatura_mediana', 'theta_v_promedio']
                variable            test   p_value        η²     ω²/ε²
1              r_mediana  Kruskal-Wallis  0.000011  0.401481  0.379717
0             r_promedio  Kruskal-Wallis  0.000011  0.400908  0.379123
2    r_centrado_promedio  Kruskal-Wallis  0.005765  0.180909  0.151124
3     r_centrado_mediana  Kruskal

### Pruebas post hoc

In [8]:
variables_tukey_cc = list(df_efectos_cc.loc[
    (df_efectos_cc['test'] == 'ANOVA') &
    (df_efectos_cc['variable'].isin(variables_candidatas_cc)),
    'variable'])

variables_gh_cc = list(df_efectos_cc.loc[
    (df_efectos_cc['test'] == "Welch's ANOVA") &
    (df_efectos_cc['variable'].isin(variables_candidatas_cc)),
    'variable'])

variables_dunn_cc = list(df_efectos_cc.loc[
    (df_efectos_cc['test'] == 'Kruskal-Wallis') &
    (df_efectos_cc['variable'].isin(variables_candidatas_cc)),
    'variable'])

In [9]:
if variables_tukey_cc:
    tukey_cc = posthoc_anova_tukey(cc, variables_tukey_cc, factor)

if variables_gh_cc:
    gh_cc = posthoc_welch_gameshowell(cc, variables_gh_cc, factor)
    
if variables_dunn_cc:
    dunn_cc = posthoc_kruskal_dunn(cc, variables_dunn_cc, factor)

In [10]:
if 'tukey_cc' in locals():
    posthocs_tukey_cc = {variable: tukey_cc.loc[tukey_cc['variable'] == variable] for variable in variables_tukey_cc}
    print("Hay df con post hoc para Tukey. Revisar")
    
if 'gh_cc' in locals():
    posthocs_gh_cc = {variable: gh_cc.loc[gh_cc['variable'] == variable] for variable in variables_gh_cc}
    print("Hay df con post hoc para GH. Revisar")
    
if 'dunn_cc' in locals():
    posthocs_dunn_cc = {variable: dunn_cc.loc[dunn_cc['variable'] == variable] for variable in variables_dunn_cc}
    print("Hay df con post hoc para Dunn. Revisar")

Hay df con post hoc para Dunn. Revisar


In [11]:
posthocs_dunn_cc

{'r_mediana':    grupo1  grupo2   p-valor   variable
 0    alta    alta  1.000000  r_mediana
 1    baja    alta  0.000009  r_mediana
 2  normal    alta  0.014213  r_mediana
 3    alta    baja  0.000009  r_mediana
 4    baja    baja  1.000000  r_mediana
 5  normal    baja  0.097341  r_mediana
 6    alta  normal  0.014213  r_mediana
 7    baja  normal  0.097341  r_mediana
 8  normal  normal  1.000000  r_mediana,
 'r_promedio':     grupo1  grupo2   p-valor    variable
 9     alta    alta  1.000000  r_promedio
 10    baja    alta  0.000009  r_promedio
 11  normal    alta  0.012463  r_promedio
 12    alta    baja  0.000009  r_promedio
 13    baja    baja  1.000000  r_promedio
 14  normal    baja  0.108541  r_promedio
 15    alta  normal  0.012463  r_promedio
 16    baja  normal  0.108541  r_promedio
 17  normal  normal  1.000000  r_promedio,
 'r_centrado_promedio':     grupo1  grupo2   p-valor             variable
 18    alta    alta  1.000000  r_centrado_promedio
 19    baja    alta  0.004

## ERIZOS

### Análisis de la varianza

In [8]:
resultados_la = df_resultados[(df_resultados['grupo'] == 'LA')]

variables_resumen_por_test = (resultados_la[['variable', 'test_sugerido']].drop_duplicates()
    .groupby('test_sugerido')['variable']
    .unique()
    .to_dict())

print("ERIZOS - QUÉ APLICAR SEGÚN SUPUESTOS")
for tipo, lista in variables_resumen_por_test.items():
    print(f"Puedo aplicar {tipo} en: {list(lista)}")
    
#filtro
listas_por_test = {
    tipo: grupo['variable'].drop_duplicates().tolist()
    for tipo, grupo in resultados_la.dropna(subset=['test_sugerido']).groupby('test_sugerido')}

#ver ranking
df_efectos_la = analizar_grupo(la, listas_por_test, 'nivel_salinidad')
pvalues_significativos_la = df_efectos_la.loc[(df_efectos_la['p_value'] < 0.05)]
variables_candidatas_la = pvalues_significativos_la['variable'].tolist()
variables_candidatas_la = set(variables_candidatas_la) #podemos mostrar el dataframe completo también
print(pvalues_significativos_la)

ERIZOS - QUÉ APLICAR SEGÚN SUPUESTOS
Puedo aplicar ANOVA en: ['theta_v_mediana']
Puedo aplicar Kruskal-Wallis en: ['r_promedio', 'r_mediana', 'r_centrado_promedio', 'r_centrado_mediana', 'v_promedio', 'v_mediana', 'a_promedio', 'a_mediana', 'dr_promedio', 'dr_mediana', 'dr_centrado_promedio', 'dr_centrado_mediana', 'theta_r_promedio', 'theta_r_mediana', 'theta_r_centrado_promedio', 'theta_r_centrado_mediana', 'v_angular_promedio', 'v_angular_mediana', 'v_angular_centrado_promedio', 'v_angular_centrado_mediana', 'a_angular_promedio', 'a_angular_mediana', 'a_angular_centrado_promedio', 'a_angular_centrado_mediana', 'curvatura_promedio', 'curvatura_mediana', 'theta_v_promedio']
     variable            test   p_value        η²     ω²/ε²
1   r_mediana  Kruskal-Wallis  0.000622  0.434244  0.398885
0  r_promedio  Kruskal-Wallis  0.000640  0.432558  0.397092


### Pruebas post hoc

In [13]:
variables_tukey_la = list(df_efectos_la.loc[
    (df_efectos_la['test'] == 'ANOVA') &
    (df_efectos_la['variable'].isin(variables_candidatas_la)),
    'variable'])

variables_gh_la = list(df_efectos_la.loc[
    (df_efectos_la['test'] == "Welch's ANOVA") &
    (df_efectos_la['variable'].isin(variables_candidatas_la)),
    'variable'])

variables_dunn_la = list(df_efectos_la.loc[
    (df_efectos_la['test'] == 'Kruskal-Wallis') &
    (df_efectos_la['variable'].isin(variables_candidatas_la)),
    'variable'])

In [14]:
if variables_tukey_la:
    tukey_la = posthoc_anova_tukey(la, variables_tukey_la, factor)

if variables_gh_la:
    gh_la = posthoc_welch_gameshowell(la, variables_gh_la, factor)
    
if variables_dunn_la:
    dunn_la = posthoc_kruskal_dunn(la, variables_dunn_la, factor)

In [15]:
if 'tukey_la' in locals():
    posthocs_tukey_la = {variable: tukey_la.loc[tukey_la['variable'] == variable] for variable in variables_tukey_la}
    print("Hay df con post hoc para Tukey. Revisar")
    
if 'gh_la' in locals():
    posthocs_gh_la = {variable: gh_la.loc[gh_la['variable'] == variable] for variable in variables_gh_la}
    print("Hay df con post hoc para GH. Revisar")
    
if 'dunn_la' in locals():
    posthocs_dunn_la = {variable: dunn_la.loc[dunn_la['variable'] == variable] for variable in variables_dunn_la}
    print("Hay df con post hoc para Dunn. Revisar")

Hay df con post hoc para Dunn. Revisar


In [16]:
posthocs_dunn_la

{'r_mediana':    grupo1  grupo2   p-valor   variable
 0    alta    alta  1.000000  r_mediana
 1    baja    alta  0.000457  r_mediana
 2  normal    alta  0.117955  r_mediana
 3    alta    baja  0.000457  r_mediana
 4    baja    baja  1.000000  r_mediana
 5  normal    baja  0.032071  r_mediana
 6    alta  normal  0.117955  r_mediana
 7    baja  normal  0.032071  r_mediana
 8  normal  normal  1.000000  r_mediana,
 'r_promedio':     grupo1  grupo2   p-valor    variable
 9     alta    alta  1.000000  r_promedio
 10    baja    alta  0.000508  r_promedio
 11  normal    alta  0.143339  r_promedio
 12    alta    baja  0.000508  r_promedio
 13    baja    baja  1.000000  r_promedio
 14  normal    baja  0.025934  r_promedio
 15    alta  normal  0.143339  r_promedio
 16    baja  normal  0.025934  r_promedio
 17  normal  normal  1.000000  r_promedio}